## **Pendekatan Klasik**

In [ ]:
!pip install Sastrawi

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import re

Load Dataset

In [ ]:
# === 1. Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# Ganti path di bawah ini dengan lokasi file kamu di Google Drive
data_path = "/content/drive/MyDrive/berita_detik_news_label.csv"

# Baca dataset
df = pd.read_csv(data_path)

# Lihat daftar kolom untuk memastikan nama kolom benar
print("Kolom dalam dataset:", df.columns.tolist())

# Pilih kolom yang digunakan (ubah sesuai nama kolom dataset kamu)
df = df[['isi', 'label']].dropna()

# Tampilkan 5 data pertama
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kolom dalam dataset: ['judul_berita', 'isi', 'date', 'url', 'label']


,isi,label
0,Menteri Koordinator Bidang Perekonomian Airlan...,Positif
1,Pemerintah Provinsi Jawa Tengah memberikan ins...,Positif
2,Prasasti Center for Policy Studies (Prasasti) ...,Netral
3,"Anggota MPR RI dari Fraksi Partai Golkar, Ahma...",Negatif
4,Dalam Rapat Paripurna DPR RI ke-5 Masa Persida...,Positif


Preprocessing Teks Bahasa Indonesia

In [ ]:
import re
import pandas as pd
from tqdm.notebook import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# === Aktifkan progress bar ===
tqdm.pandas()

# Inisialisasi stemmer & stopword remover
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# === Pilihan Cepat / Lengkap ===
USE_STEMMING = False  # ubah ke True jika ingin pakai stemming penuh (lebih lama)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)           # hapus simbol, angka
    tokens = text.split()
    tokens = [w for w in tokens if w not in stopwords]
    text = " ".join(tokens)

    if USE_STEMMING:
        text = stemmer.stem(text)                  # aktifkan hanya jika perlu
    return text

TF-IDF Vectorization

In [ ]:
# Pisahkan fitur (X) dan label (y)
X = df['clean_text']
y = df['label']

# Bagi data untuk training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Konversi teks menjadi vektor TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Jumlah fitur TF-IDF:", len(vectorizer.get_feature_names_out()))

Jumlah fitur TF-IDF: 5000


Training Model Naive Bayes

In [ ]:
# Buat dan latih model
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

# Prediksi data test
y_pred = nb.predict(X_test_tfidf)

Evaluasi Model

In [ ]:
print("=== HASIL EVALUASI MODEL KLASIK ===")
print("Akurasi :", round(accuracy_score(y_test, y_pred)*100, 2), "%\n")

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== HASIL EVALUASI MODEL KLASIK ===
Akurasi : 76.87 %

=== Classification Report ===
              precision    recall  f1-score   support

     Negatif       0.00      0.00      0.00       147
      Netral       0.77      1.00      0.87       812
     Positif       0.00      0.00      0.00        96

    accuracy                           0.77      1055
   macro avg       0.26      0.33      0.29      1055
weighted avg       0.59      0.77      0.67      1055

=== Confusion Matrix ===
[[  0 147   0]
 [  0 811   1]
 [  0  96   0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model SVM

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

In [ ]:
print("\n=== Support Vector Machine ===")
print("Akurasi:", accuracy_score(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


=== Support Vector Machine ===
Akurasi: 0.7497630331753554
[[  0 154   0]
 [  4 791   1]
 [  0 105   0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       154
           1       0.75      0.99      0.86       796
           2       0.00      0.00      0.00       105

    accuracy                           0.75      1055
   macro avg       0.25      0.33      0.29      1055
weighted avg       0.57      0.75      0.65      1055



Model rendome Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
print("\n=== Random Forest ===")
print("Akurasi:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


=== Random Forest ===
Akurasi: 0.7260663507109004
[[  9 144   1]
 [ 26 756  14]
 [  1 103   1]]
              precision    recall  f1-score   support

           0       0.25      0.06      0.09       154
           1       0.75      0.95      0.84       796
           2       0.06      0.01      0.02       105

    accuracy                           0.73      1055
   macro avg       0.36      0.34      0.32      1055
weighted avg       0.61      0.73      0.65      1055



BoW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2))
X_bow = vectorizer.fit_transform(X)

Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [ ]:
print("\n=== [Naive Bayes - BoW] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))


=== [Naive Bayes - BoW] ===
Akurasi : 0.35165876777251187
Confusion Matrix:
 [[ 63  60  31]
 [292 288 216]
 [ 39  46  20]]
Classification Report:
               precision    recall  f1-score   support

           0       0.16      0.41      0.23       154
           1       0.73      0.36      0.48       796
           2       0.07      0.19      0.11       105

    accuracy                           0.35      1055
   macro avg       0.32      0.32      0.27      1055
weighted avg       0.58      0.35      0.41      1055



 Support Vector Machine

In [ ]:
svm = LinearSVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

In [ ]:
print("\n=== [Support Vector Machine - BoW] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))


=== [Support Vector Machine - BoW] ===
Akurasi : 0.7497630331753554
Confusion Matrix:
 [[  0 154   0]
 [  4 791   1]
 [  0 105   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       154
           1       0.75      0.99      0.86       796
           2       0.00      0.00      0.00       105

    accuracy                           0.75      1055
   macro avg       0.25      0.33      0.29      1055
weighted avg       0.57      0.75      0.65      1055



Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
print("\n=== [Random Forest - BoW] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


=== [Random Forest - BoW] ===
Akurasi : 0.7260663507109004
Confusion Matrix:
 [[  9 144   1]
 [ 26 756  14]
 [  1 103   1]]
Classification Report:
               precision    recall  f1-score   support

           0       0.25      0.06      0.09       154
           1       0.75      0.95      0.84       796
           2       0.06      0.01      0.02       105

    accuracy                           0.73      1055
   macro avg       0.36      0.34      0.32      1055
weighted avg       0.61      0.73      0.65      1055



## Deep Learning

In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/berita_detik_news_label.csv')
X = df['isi']
y = df['label']

Pretrained FastText Embedding

In [ ]:
print("🔄 Mengunduh FastText pretrained embedding...")
fasttext_model = api.load("fasttext-wiki-news-subwords-300")

🔄 Mengunduh FastText pretrained embedding...
[==================================================] 100.0% 958.5/958.4MB downloaded


ubah teks jadi vektor rata-rata FastText

In [ ]:
X = X.fillna("").astype(str)

def get_fasttext_vector(text):
    words = text.split()
    word_vecs = [fasttext_model[word] for word in words if word in fasttext_model]
    if len(word_vecs) == 0:
        return np.zeros(300)
    return np.mean(word_vecs, axis=0)

print("Mengubah teks menjadi vektor FastText")
X_vec = np.vstack(X.apply(get_fasttext_vector))
print("Selesai membuat representasi vektor FastText!")

Mengubah teks menjadi vektor FastText
Selesai membuat representasi vektor FastText!


Split dataset

In [ ]:
X_vec = np.nan_to_num(X_vec, nan=0.0)

# Pastikan label tidak ada NaN juga
y = y.fillna(method='ffill')  # isi label kosong dengan nilai sebelumnya (jika ada)

# Sekarang bisa dipisah
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

/tmp/ipython-input-4155808085.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  y = y.fillna(method='ffill')  # isi label kosong dengan nilai sebelumnya (jika ada)


Naive Bayes

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("\n=== [Naive Bayes - FastText] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))


=== [Naive Bayes - FastText] ===
Akurasi : 0.32169811320754715
Confusion Matrix:
 [[ 26  48  81]
 [102 259 438]
 [ 11  39  56]]
Classification Report:
               precision    recall  f1-score   support

     Negatif       0.19      0.17      0.18       155
      Netral       0.75      0.32      0.45       799
     Positif       0.10      0.53      0.16       106

    accuracy                           0.32      1060
   macro avg       0.34      0.34      0.26      1060
weighted avg       0.60      0.32      0.38      1060



Support Vector Machine

In [ ]:
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("\n=== [SVM - FastText] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))


=== [SVM - FastText] ===
Akurasi : 0.7537735849056604
Confusion Matrix:
 [[  0 155   0]
 [  0 799   0]
 [  0 106   0]]
Classification Report:
               precision    recall  f1-score   support

     Negatif       0.00      0.00      0.00       155
      Netral       0.75      1.00      0.86       799
     Positif       0.00      0.00      0.00       106

    accuracy                           0.75      1060
   macro avg       0.25      0.33      0.29      1060
weighted avg       0.57      0.75      0.65      1060



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n=== [Random Forest - FastText] ===")
print("Akurasi :", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))


=== [Random Forest - FastText] ===
Akurasi : 0.7386792452830189
Confusion Matrix:
 [[ 10 143   2]
 [ 19 769  11]
 [  1 101   4]]
Classification Report:
               precision    recall  f1-score   support

     Negatif       0.33      0.06      0.11       155
      Netral       0.76      0.96      0.85       799
     Positif       0.24      0.04      0.07       106

    accuracy                           0.74      1060
   macro avg       0.44      0.35      0.34      1060
weighted avg       0.64      0.74      0.66      1060



## Transfer Learning

In [ ]:
!pip install transformers torch scikit-learn tqdm -q

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
X = df['isi']
y = df['label']

IndoBERT

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Konversi Teks → Embedding BERT

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
bert_model.eval()

def get_bert_embedding(text):
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)
        # Ambil vektor [CLS] (representasi keseluruhan kalimat)
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return cls_embedding.flatten()

# Gunakan tqdm untuk progress
print("🔄 Mengubah teks menjadi embedding BERT...")
X_vec = np.vstack([get_bert_embedding(str(t)) for t in tqdm(X)])

🔄 Mengubah teks menjadi embedding BERT...


  0%|          | 0/5300 [00:00<?, ?it/s]

Split Data

In [ ]:
import numpy as np
import pandas as pd

# Pastikan tidak ada NaN di label
df = df.dropna(subset=['isi', 'label']).reset_index(drop=True)
y = df['label']

# Pastikan ukuran X_vec dan y cocok
if X_vec.shape[0] != len(y):
    print(f"⚠️ Ukuran tidak cocok: X_vec = {X_vec.shape[0]}, y = {len(y)}")
    min_len = min(X_vec.shape[0], len(y))
    X_vec = X_vec[:min_len]
    y = y[:min_len]

# Cek apakah ada NaN di dalam embedding
nan_count = np.isnan(X_vec).sum()
if nan_count > 0:
    print(f"⚠️ Terdeteksi {nan_count} nilai NaN di embedding, mengganti dengan 0.")
    X_vec = np.nan_to_num(X_vec)  # mengganti semua NaN dengan nol

# Konfirmasi hasil bersih
print("✅ Tidak ada NaN lagi di X_vec:", np.isnan(X_vec).sum() == 0)
print("Ukuran data akhir:", X_vec.shape, len(y))

⚠️ Ukuran tidak cocok: X_vec = 5300, y = 5274
✅ Tidak ada NaN lagi di X_vec: True
Ukuran data akhir: (5274, 768) 5274


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)

Naive Bayes

In [ ]:
print("\nTraining Naive Bayes...")
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Accuracy (NB):", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


Training Naive Bayes...
Accuracy (NB): 0.5194312796208531
              precision    recall  f1-score   support

     Negatif       0.23      0.19      0.21       154
      Netral       0.76      0.61      0.67       796
     Positif       0.12      0.34      0.18       105

    accuracy                           0.52      1055
   macro avg       0.37      0.38      0.35      1055
weighted avg       0.62      0.52      0.56      1055



SVM

In [ ]:
print("\n⚙️ Training SVM...")
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
print("Accuracy (SVM):", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


⚙️ Training SVM...
Accuracy (SVM): 0.7014218009478673
              precision    recall  f1-score   support

     Negatif       0.17      0.06      0.09       154
      Netral       0.77      0.91      0.83       796
     Positif       0.17      0.09      0.11       105

    accuracy                           0.70      1055
   macro avg       0.37      0.35      0.35      1055
weighted avg       0.62      0.70      0.65      1055



Random Forest

In [ ]:
print("\n🌲 Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print("Accuracy (RF):", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


🌲 Training Random Forest...
Accuracy (RF): 0.7317535545023697
              precision    recall  f1-score   support

     Negatif       0.19      0.04      0.06       154
      Netral       0.76      0.96      0.85       796
     Positif       0.08      0.01      0.02       105

    accuracy                           0.73      1055
   macro avg       0.34      0.34      0.31      1055
weighted avg       0.61      0.73      0.65      1055

